# EduFlowTech: Sistema multiagente para la gestión de tickets con LangChain


## 1. Preparación del entorno


In [1]:
%pip install -q --upgrade langchain langchain-openai pandas pydantic


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 164.7/164.7 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 52.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 572.1/572.1 kB 18.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.0.6 which is incompatible.


In [2]:
import json
import os
import re
import unicodedata
from typing import Any, Dict, List, Literal, Optional

import pandas as pd
from pydantic import BaseModel, Field, ValidationError
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda
from langchain_openai import ChatOpenAI

# Fake key required by the exercise. Never upload a real API key to GitHub.
os.environ["OPENAI_API_KEY"] = "sk-YOUR_FAKE_API_KEY"
MODEL_NAME = "gpt-4o-mini"
USE_LLM = False  # Change to True only after configuring a valid key locally.


## 2. Diagrama del flujo de trabajo

```mermaid
flowchart LR
    U[User support request] --> A1[Agent 1: Query Processor]
    A1 -->|Structured intent, topic, level and issue details| A2[Agent 2: Content Search]
    DB[(Simulated EduFlowTech database)] --> A2
    A2 -->|Ranked lessons, exercises or bug records| A3[Agent 3: Response Generator]
    A1 --> A3
    A3 --> R[Personalised answer and escalation status]
```

## 3. Base de datos simulada de EduFlowTech



In [3]:
CATALOG_DATA = [
    {
        "record_id": "LES-101", "record_type": "lesson", "course": "Fundamentos de Deep Learning",
        "topic": "neural networks perceptrons activation functions", "title": "Introducción a las Redes Neuronales",
        "exercise": "Construir un perceptrón para clasificación binaria", "difficulty": "beginner",
        "content_text": "Fundamentos de neuronas, capas, pesos, sesgos y funciones de activación.",
        "url": "https://example.com/eduflowtech/deep-learning/neural-networks-intro",
        "image_url": "https://example.com/images/neural-network-diagram.png",
        "image_description": "Diagrama de una capa de entrada, una capa oculta y una capa de salida",
        "estimated_minutes": 35
    },
    {
        "record_id": "LES-102", "record_type": "lesson", "course": "Fundamentos de Deep Learning",
        "topic": "backpropagation gradient descent training", "title": "Entrenamiento de una Red Neuronal",
        "exercise": "Calcular una actualización mediante retropropagación", "difficulty": "intermediate",
        "content_text": "Funciones de pérdida, descenso de gradiente y retropropagación con un ejemplo guiado.",
        "url": "https://example.com/eduflowtech/deep-learning/backpropagation",
        "image_url": "https://example.com/images/backpropagation-flow.png",
        "image_description": "Diagrama de flujo de la propagación hacia delante y hacia atrás",
        "estimated_minutes": 50
    },
    {
        "record_id": "EX-201", "record_type": "exercise", "course": "Python para Análisis de Datos",
        "topic": "pandas dataframe filtering missing values", "title": "Reto de Limpieza de DataFrames",
        "exercise": "Limpiar valores ausentes y filtrar filas de ventas", "difficulty": "beginner",
        "content_text": "Práctica de pandas con fillna, dropna, loc y filtros booleanos.",
        "url": "https://example.com/eduflowtech/python/pandas-cleaning-exercise",
        "image_url": "https://example.com/images/dataframe-cleaning.png",
        "image_description": "Tabla que resalta celdas ausentes y filas filtradas",
        "estimated_minutes": 25
    },
    {
        "record_id": "LES-301", "record_type": "lesson", "course": "Procesamiento del Lenguaje Natural",
        "topic": "sentiment analysis text classification transformers", "title": "Clasificación de Sentimiento",
        "exercise": "Clasificar opiniones de clientes como positivas, neutras o negativas", "difficulty": "intermediate",
        "content_text": "Preprocesamiento de texto, embeddings de transformers y etiquetas de sentimiento.",
        "url": "https://example.com/eduflowtech/nlp/sentiment-classification",
        "image_url": "https://example.com/images/sentiment-dashboard.png",
        "image_description": "Panel con indicadores de sentimiento positivo, neutro y negativo",
        "estimated_minutes": 45
    },
    {
        "record_id": "BUG-401", "record_type": "bug", "course": "Soporte de la Plataforma",
        "topic": "video playback black screen browser", "title": "El vídeo muestra una pantalla negra",
        "exercise": "No aplicable", "difficulty": "not_applicable",
        "content_text": "Comprobaciones recomendadas: actualizar la página, verificar la compatibilidad del navegador, desactivar extensiones que puedan causar conflictos y volver a intentarlo.",
        "url": "https://example.com/eduflowtech/support/video-black-screen",
        "image_url": "https://example.com/images/video-error-example.png",
        "image_description": "Ejemplo de la pantalla negra del reproductor de vídeo",
        "estimated_minutes": 5
    },
    {
        "record_id": "BUG-402", "record_type": "bug", "course": "Soporte de la Plataforma",
        "topic": "exercise submission upload error file format", "title": "La entrega del ejercicio falla",
        "exercise": "No aplicable", "difficulty": "not_applicable",
        "content_text": "Comprueba el tamaño del archivo, el formato admitido, la estabilidad de la conexión y si el plazo de entrega sigue abierto.",
        "url": "https://example.com/eduflowtech/support/submission-error",
        "image_url": "https://example.com/images/submission-error.png",
        "image_description": "Formulario de entrega con una notificación de error de carga",
        "estimated_minutes": 7
    }
]

catalog_df = pd.DataFrame(CATALOG_DATA)
catalog_df


,record_id,record_type,course,topic,title,exercise,difficulty,content_text,url,image_url,image_description,estimated_minutes
0,LES-101,lesson,Fundamentos de Deep Learning,neural networks perceptrons activation functions,Introducción a las Redes Neuronales,Construir un perceptrón para clasificación bin...,beginner,"Fundamentos de neuronas, capas, pesos, sesgos ...",https://example.com/eduflowtech/deep-learning/...,https://example.com/images/neural-network-diag...,"Diagrama de una capa de entrada, una capa ocul...",35
1,LES-102,lesson,Fundamentos de Deep Learning,backpropagation gradient descent training,Entrenamiento de una Red Neuronal,Calcular una actualización mediante retropropa...,intermediate,"Funciones de pérdida, descenso de gradiente y ...",https://example.com/eduflowtech/deep-learning/...,https://example.com/images/backpropagation-flo...,Diagrama de flujo de la propagación hacia dela...,50
2,EX-201,exercise,Python para Análisis de Datos,pandas dataframe filtering missing values,Reto de Limpieza de DataFrames,Limpiar valores ausentes y filtrar filas de ve...,beginner,"Práctica de pandas con fillna, dropna, loc y f...",https://example.com/eduflowtech/python/pandas-...,https://example.com/images/dataframe-cleaning.png,Tabla que resalta celdas ausentes y filas filt...,25
3,LES-301,lesson,Procesamiento del Lenguaje Natural,sentiment analysis text classification transfo...,Clasificación de Sentimiento,Clasificar opiniones de clientes como positiva...,intermediate,"Preprocesamiento de texto, embeddings de trans...",https://example.com/eduflowtech/nlp/sentiment-...,https://example.com/images/sentiment-dashboard...,"Panel con indicadores de sentimiento positivo,...",45
4,BUG-401,bug,Soporte de la Plataforma,video playback black screen browser,El vídeo muestra una pantalla negra,No aplicable,not_applicable,Comprobaciones recomendadas: actualizar la pág...,https://example.com/eduflowtech/support/video-...,https://example.com/images/video-error-example...,Ejemplo de la pantalla negra del reproductor d...,5
5,BUG-402,bug,Soporte de la Plataforma,exercise submission upload error file format,La entrega del ejercicio falla,No aplicable,not_applicable,"Comprueba el tamaño del archivo, el formato ad...",https://example.com/eduflowtech/support/submis...,https://example.com/images/submission-error.png,Formulario de entrega con una notificación de ...,7


## 4. Esquemas compartidos y funciones auxiliares


In [4]:
class ProcessedQuery(BaseModel):
    category: Literal["lesson", "exercise", "bug", "general"]
    topic: str
    difficulty: Optional[Literal["beginner", "intermediate", "advanced"]] = None
    user_type: Literal["student", "teacher", "administrator", "unknown"] = "unknown"
    keywords: List[str]
    needs_escalation: bool = False


class SearchResult(BaseModel):
    record_id: str
    record_type: str
    title: str
    course: str
    difficulty: str
    content_text: str
    exercise: str
    url: str
    image_url: str
    image_description: str
    estimated_minutes: int
    relevance_score: float = Field(ge=0, le=1)


class SupportResponse(BaseModel):
    category: str
    answer: str
    recommended_title: Optional[str] = None
    recommended_url: Optional[str] = None
    next_steps: List[str]
    escalation_required: bool
    confidence: float = Field(ge=0, le=1)


def normalise(text: str) -> str:
    text = unicodedata.normalize("NFKD", text).encode("ascii", "ignore").decode().lower()
    return re.sub(r"[^a-z0-9 ]+", " ", text)


def tokens(text: str) -> set:
    stop = {"the", "a", "an", "to", "for", "about", "on", "of", "i", "my", "is", "are", "how", "what", "which", "with", "and", "de", "la", "el", "los", "las", "para", "un", "una", "que", "como", "sobre"}
    return {w for w in normalise(text).split() if len(w) > 2 and w not in stop}


## 5. Agente 1: Procesador de consultas


In [5]:
class QueryProcessorAgent:
    def __init__(self, use_llm: bool = False):
        self.use_llm = use_llm
        self.prompt = ChatPromptTemplate.from_messages([
            ("system", "Eres el agente de clasificación de consultas de EduFlowTech. Clasifica la solicitud, extrae el tema, tipo de usuario, dificultad y necesidad de escalado. Devuelve siempre la información estructurada en español."),
            ("human", "Consulta del usuario: {query}")
        ])
        if use_llm:
            model = ChatOpenAI(model=MODEL_NAME, temperature=0)
            self.chain = self.prompt | model.with_structured_output(ProcessedQuery)

    def invoke(self, query: str) -> ProcessedQuery:
        if self.use_llm:
            return self.chain.invoke({"query": query})

        q = normalise(query)
        if any(x in q for x in ["bug", "error", "fails", "failed", "black screen", "not working", "no funciona", "pantalla negra", "falla"]):
            category = "bug"
        elif any(x in q for x in ["exercise", "practice", "challenge", "ejercicio", "practicar"]):
            category = "exercise"
        elif any(x in q for x in ["lesson", "course", "learn", "leccion", "curso", "aprender"]):
            category = "lesson"
        else:
            category = "general"

        difficulty = None
        for level in ["beginner", "intermediate", "advanced"]:
            if level in q:
                difficulty = level
        if "principiante" in q: difficulty = "beginner"
        if "intermedio" in q: difficulty = "intermediate"
        if "avanzado" in q: difficulty = "advanced"

        user_type = "unknown"
        if any(x in q for x in ["student", "alumno", "estudiante"]): user_type = "student"
        elif any(x in q for x in ["teacher", "profesor", "docente"]): user_type = "teacher"
        elif any(x in q for x in ["administrator", "admin", "administrador"]): user_type = "administrator"

        urgent_terms = ["data loss", "security", "payment", "lost all", "perdida de datos", "seguridad", "pago"]
        return ProcessedQuery(
            category=category,
            topic=query,
            difficulty=difficulty,
            user_type=user_type,
            keywords=sorted(tokens(query)),
            needs_escalation=any(term in q for term in urgent_terms)
        )


## 6. Agente 2: Buscador de contenido



In [6]:
class ContentSearchAgent:
    def __init__(self, database: pd.DataFrame):
        self.database = database.copy()

    def invoke(self, processed: ProcessedQuery, top_k: int = 3) -> List[SearchResult]:
        db = self.database.copy()
        if processed.category in {"lesson", "exercise", "bug"}:
            preferred = db[db["record_type"] == processed.category]
            if not preferred.empty:
                db = preferred
        if processed.difficulty and processed.category != "bug":
            level_matches = db[db["difficulty"] == processed.difficulty]
            if not level_matches.empty:
                db = level_matches

        query_tokens = set(processed.keywords)
        scored = []
        for _, row in db.iterrows():
            searchable = " ".join(str(row[c]) for c in ["topic", "title", "course", "content_text", "exercise", "image_description"])
            doc_tokens = tokens(searchable)
            overlap = len(query_tokens & doc_tokens)
            score = overlap / max(len(query_tokens), 1)
            if processed.category == row["record_type"]:
                score += 0.20
            score = min(round(score, 3), 1.0)
            scored.append((score, row))

        scored.sort(key=lambda item: item[0], reverse=True)
        results = []
        for score, row in scored[:top_k]:
            payload = row.to_dict()
            payload["relevance_score"] = score
            results.append(SearchResult(**payload))
        return results


## 7. Agente 3: Generador de respuestas personalizadas



In [7]:
class ResponseGeneratorAgent:
    def __init__(self, use_llm: bool = False):
        self.use_llm = use_llm
        self.prompt = ChatPromptTemplate.from_messages([
            ("system", "Eres el agente de soporte de EduFlowTech. Responde siempre en español. Utiliza únicamente el análisis y los resultados proporcionados. Personaliza la respuesta según el tipo de usuario, sé claro, no inventes enlaces y señala si la incidencia debe escalarse."),
            ("human", "Análisis de la consulta: {processed}\nResultados encontrados: {results}")
        ])
        if use_llm:
            model = ChatOpenAI(model=MODEL_NAME, temperature=0.2)
            self.chain = self.prompt | model.with_structured_output(SupportResponse)

    def invoke(self, processed: ProcessedQuery, results: List[SearchResult]) -> SupportResponse:
        if self.use_llm:
            return self.chain.invoke({
                "processed": processed.model_dump_json(),
                "results": json.dumps([r.model_dump() for r in results], ensure_ascii=False)
            })

        best = results[0] if results else None
        low_confidence = best is None or best.relevance_score < 0.25
        escalation = processed.needs_escalation or (processed.category == "bug" and low_confidence)

        if low_confidence:
            return SupportResponse(
                category=processed.category,
                answer="No he encontrado un contenido suficientemente relevante en el catálogo de EduFlowTech. Indica el nombre del curso, el mensaje de error exacto o el nivel de dificultad que buscas.",
                next_steps=["Aportar más contexto", "Indicar el curso o la lección", "Adjuntar el texto exacto del error o describir la captura de pantalla"],
                escalation_required=escalation,
                confidence=0.20
            )

        greeting = {
            "student": "Para tu objetivo de aprendizaje",
            "teacher": "Para tus necesidades docentes",
            "administrator": "Para la administración de la plataforma",
            "unknown": "Según tu consulta"
        }[processed.user_type]

        if processed.category == "bug":
            answer = f"{greeting}, el artículo de soporte más relacionado es '{best.title}'. {best.content_text}"
            steps = ["Abrir el artículo de soporte", "Probar cada comprobación indicada", "Anotar el error exacto si el problema continúa"]
        elif processed.category == "exercise":
            answer = f"{greeting}, te recomiendo el ejercicio '{best.title}' del curso '{best.course}'. Se centra en: {best.exercise}. Tiempo estimado: {best.estimated_minutes} minutos."
            steps = ["Revisar la lección relacionada", "Completar el ejercicio", "Comparar el resultado con el objetivo de aprendizaje"]
        else:
            answer = f"{greeting}, la lección recomendada es '{best.title}' del curso '{best.course}'. {best.content_text} Tiempo estimado: {best.estimated_minutes} minutos."
            steps = ["Abrir la lección recomendada", "Revisar el recurso visual", "Completar el ejercicio asociado"]

        return SupportResponse(
            category=processed.category,
            answer=answer,
            recommended_title=best.title,
            recommended_url=best.url,
            next_steps=steps,
            escalation_required=escalation,
            confidence=max(0.55, best.relevance_score)
        )


## 8. Orquestación del flujo con LangChain



In [8]:
processor_agent = QueryProcessorAgent(use_llm=USE_LLM)
search_agent = ContentSearchAgent(catalog_df)
response_agent = ResponseGeneratorAgent(use_llm=USE_LLM)


def processor_step(state: Dict[str, Any]) -> Dict[str, Any]:
    state["processed_query"] = processor_agent.invoke(state["query"])
    return state


def search_step(state: Dict[str, Any]) -> Dict[str, Any]:
    state["search_results"] = search_agent.invoke(state["processed_query"])
    return state


def response_step(state: Dict[str, Any]) -> Dict[str, Any]:
    state["final_response"] = response_agent.invoke(state["processed_query"], state["search_results"])
    return state


support_workflow = (
    RunnableLambda(processor_step)
    | RunnableLambda(search_step)
    | RunnableLambda(response_step)
)


def run_ticket(query: str) -> Dict[str, Any]:
    try:
        return support_workflow.invoke({"query": query})
    except ValidationError as exc:
        return {"query": query, "error": "Schema validation failed", "details": exc.errors()}
    except Exception as exc:
        return {"query": query, "error": type(exc).__name__, "details": str(exc)}


## 9. Ejemplo práctico obligatorio


In [10]:
example = run_ticket("¿Cuál es la lección más adecuada para aprender sobre redes neuronales? Soy estudiante principiante.")

print("AGENTE 1 - CONSULTA PROCESADA")
print(example["processed_query"].model_dump_json(indent=2))
print("AGENTE 2 - RESULTADOS DE BÚSQUEDA")
print(json.dumps([x.model_dump() for x in example["search_results"]], ensure_ascii=False, indent=2))
print("AGENTE 3 - RESPUESTA FINAL")
print(example["final_response"].model_dump_json(indent=2))


AGENTE 1 - CONSULTA PROCESADA
{
  "category": "lesson",
  "topic": "¿Cuál es la lección más adecuada para aprender sobre redes neuronales? Soy estudiante principiante.",
  "difficulty": "beginner",
  "user_type": "student",
  "keywords": [
    "adecuada",
    "aprender",
    "cual",
    "estudiante",
    "leccion",
    "mas",
    "neuronales",
    "principiante",
    "redes",
    "soy"
  ],
  "needs_escalation": false
}
AGENTE 2 - RESULTADOS DE BÚSQUEDA
[
  {
    "record_id": "LES-101",
    "record_type": "lesson",
    "title": "Introducción a las Redes Neuronales",
    "course": "Fundamentos de Deep Learning",
    "difficulty": "beginner",
    "content_text": "Fundamentos de neuronas, capas, pesos, sesgos y funciones de activación.",
    "exercise": "Construir un perceptrón para clasificación binaria",
    "url": "https://example.com/eduflowtech/deep-learning/neural-networks-intro",
    "image_url": "https://example.com/images/neural-network-diagram.png",
    "image_description": "Dia

## 10. Pruebas con diferentes escenarios

The following tests cover a lesson, an exercise, two bugs and an unknown request. This demonstrates adaptation and error detection.


In [11]:
TEST_QUERIES = [
    "Soy estudiante principiante. ¿Qué lección debería usar para aprender redes neuronales?",
    "Necesito un ejercicio para principiantes con el que practicar la limpieza de valores ausentes en pandas.",
    "El vídeo del curso muestra una pantalla negra y no funciona.",
    "La entrega de mi ejercicio falla cuando intento subir el archivo.",
    "¿Puedes ayudarme con jardinería cuántica para robots submarinos?"
]

rows = []
for query in TEST_QUERIES:
    output = run_ticket(query)
    if "error" in output:
        rows.append({"query": query, "status": "error", "category": None, "recommendation": None, "answer": output["details"]})
    else:
        response = output["final_response"]
        rows.append({
            "query": query,
            "status": "ok",
            "category": output["processed_query"].category,
            "recommendation": response.recommended_title,
            "answer": response.answer,
            "escalation": response.escalation_required,
            "confidence": response.confidence
        })

test_results_df = pd.DataFrame(rows)
test_results_df


,query,status,category,recommendation,answer,escalation,confidence
0,Soy estudiante principiante. ¿Qué lección debe...,ok,lesson,Introducción a las Redes Neuronales,"Para tu objetivo de aprendizaje, la lección re...",False,0.550
1,Necesito un ejercicio para principiantes con e...,ok,exercise,Reto de Limpieza de DataFrames,"Según tu consulta, te recomiendo el ejercicio ...",False,0.756
2,El vídeo del curso muestra una pantalla negra ...,ok,bug,El vídeo muestra una pantalla negra,"Según tu consulta, el artículo de soporte más ...",False,0.914
3,La entrega de mi ejercicio falla cuando intent...,ok,bug,La entrega del ejercicio falla,"Según tu consulta, el artículo de soporte más ...",False,0.771
4,¿Puedes ayudarme con jardinería cuántica para ...,ok,general,NaN,No he encontrado un contenido suficientemente ...,False,0.200
